<div style="text-align: right"> <font color='Gray'> Laboratorio IV - 2026 </div>
<div style="text-align: right"> <font color='Gray'> Guía Choroplet e importación </div>
<div style="text-align: right"> <font color='Gray'> Sebastián Pulgares </div>


***

# Imports

In [ ]:
import json
import os
import numpy as np
import geopandas as gpd
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go 

# Data

In [ ]:
geodata_aldeas = gpd.read_file('../data/raw/Comunas/Cartografia_censo2024_Pais.gpkg')

### Como esta data tiene varias capas, seleccionamos las de interés

In [ ]:
geodata_aldeas = gpd.read_file("../data/raw/Comunas/Cartografia_censo2024_Pais.gpkg", layer = "Aldeas_CPV24")
#geodata_aldeas = geodata_aldeas.dropna(subset=["n_per"])
geodata_entidades = gpd.read_file("../data/raw/Comunas/Cartografia_censo2024_Pais.gpkg", layer = "Entidades_CPV24")
#geodata_entidades = geodata_entidades.dropna(subset=["n_per"])
geodata_manzanas =  gpd.read_file("../data/raw/Comunas/Cartografia_censo2024_Pais.gpkg", layer = "Manzanas_CPV24")
#geodata_manzanas = geodata_entidades.dropna(subset=["n_per"])

### Filtrado

In [ ]:
for i in geodata_manzanas.columns:
    print(i)

In [ ]:
columna_interes = 'n_hombres'

geodata_entidades_lugar = geodata_entidades[geodata_entidades['COMUNA'] == 'CONCEPCIÓN'].copy()
geodata_entidades_lugar['INTERES'] = geodata_entidades_lugar[columna_interes]/geodata_entidades_lugar['n_per']

#geodata_aldeas_lugar = geodata_aldeas[geodata_aldeas['COMUNA'] == 'SANTA CRUZ']
#geodata_aldeas_lugar['INTERES'] = geodata_aldeas_lugar[columna_interes]/geodata_aldeas_lugar['n_per']

geodata_manzanas_lugar = geodata_manzanas[geodata_manzanas['COMUNA'] == 'CONCEPCIÓN'].copy()
geodata_manzanas_lugar['INTERES'] = geodata_manzanas_lugar[columna_interes]/geodata_manzanas_lugar['n_per']

# Gráficos

In [ ]:
fig, ax = plt.subplots(figsize=(20, 20))
geodata_manzanas_lugar.plot(
    ax=ax,
    column="n_per",
    #cmap='Purples',
    legend=True,
    edgecolor='black'
)

geodata_entidades_lugar.plot(
    ax=ax,
    column="n_per",
    #cmap="Set1",
    #facecolor="none",
    linewidth=2.0,
    edgecolor="black",
    zorder=2
)
ax.set_xticks([])
ax.set_yticks([])
#plt.savefig('Prueba manzanas')
plt.show()

In [ ]:
geodata_manzanas_test = geodata_manzanas[geodata_manzanas["COMUNA"] == "CONCEPCIÓN"].copy()
geodata_entidades_test = geodata_entidades[geodata_entidades["COMUNA"] == "CONCEPCIÓN"].copy()

manzanas = geodata_manzanas_test.to_crs(4326).reset_index(drop=True).copy()
entidades = geodata_entidades_test.to_crs(4326).reset_index(drop=True).copy()

entidades["ratio"] = (
    entidades["n_hombres"].fillna(0) / entidades["n_per"].replace(0, np.nan)
).fillna(0)

manzanas["ratio"] = (
    manzanas["n_hombres"].fillna(0) / manzanas["n_per"].replace(0, np.nan)
).fillna(0)

vmin = np.nanmin([0, 1])
vmax = np.nanmax([0, 1])

manzanas["_id"] = manzanas.index.astype(str)
entidades["_id"] = entidades.index.astype(str)

manzanas_geojson = json.loads(manzanas.to_json())
entidades_geojson = json.loads(entidades.to_json())

visualization_path = os.path.join("..", "visualizations", "concepcion.html")

fig = go.Figure()

# Entidades
fig.add_trace(
    go.Choropleth(
        geojson=entidades_geojson,
        locations=entidades["_id"],
        featureidkey="properties._id",
        z=entidades["ratio"],
        coloraxis="coloraxis",
        zmin=vmin,
        zmax=vmax,
        marker_line_color="black",
        marker_line_width=2.0,
        customdata=entidades[["LOCALIDAD"]].fillna("SIN LOCALIDAD"),
        hovertemplate="<b>%{customdata[0]}</b><br>proportion: %{z:.4f}<extra></extra>",
        name="Entidades",
    )
)

# Manzanas
fig.add_trace(
    go.Choropleth(
        geojson=manzanas_geojson,
        locations=manzanas["_id"],
        featureidkey="properties._id",
        z=manzanas["ratio"],
        coloraxis="coloraxis",
        zmin=vmin,
        zmax=vmax,
        marker_line_color="white",
        marker_line_width=0.2,
        customdata=manzanas[["LOCALIDAD"]].fillna("SIN LOCALIDAD"),
        hovertemplate="<b>%{customdata[0]}</b><br>proportion: %{z:.4f}<extra></extra>",
        name="Manzanas",
    )
)

fig.update_geos(fitbounds="locations", visible=False, projection_type="mercator")

fig.update_layout(
    width=1000,
    height=750,
    margin=dict(l=0, r=0, t=40, b=0),
    coloraxis=dict(
        colorscale="Viridis",
        cmin=vmin,
        cmax=vmax,
        colorbar=dict(title="proportion"),
    ),
)

fig.show()
fig.write_html(
    visualization_path,
    full_html=True,
    include_plotlyjs=True
)